# 05 - Scenario simulation & recommendations
For every product, simulate shipping from each alternative factory, predict the new lead time, measure profit sensitivity, rank by lead-time reduction / risk reduction / profit impact, and generate top-N recommendations.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]          # notebooks/<stage>/ -> project root
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import config as C
from src import modeling
from src.simulation import ScenarioSimulator
from src.optimization import score_scenarios, recommend, alternatives_per_product, network_kpis

In [2]:
prepared = pd.read_csv(C.PROCESSED / 'orders_prepared.csv', parse_dates=['Order Date','Ship Date'])
bundle = modeling.load_bundle(prepared)
sim = ScenarioSimulator(prepared, bundle)
scored = score_scenarios(sim.simulate(), speed_weight=0.5)
scored[scored['Product']=='Wonka Bar - Milk Chocolate'][['Candidate Factory','Scenario Lead (days)','Lead Reduction %','Distance Reduction %','Profit Impact %','Confidence','Risk Level','Score']].round(2)

,Candidate Factory,Scenario Lead (days),Lead Reduction %,Distance Reduction %,Profit Impact %,Confidence,Risk Level,Score
15,Lot's O' Nuts,4.72,-1.32,-13.53,-0.58,81.14,Medium,-0.06
16,Wicked Choccy's,4.66,0.00,0.00,0.00,90.14,-,0.00
17,Sugar Shack,4.61,0.93,5.14,0.30,72.14,Medium,0.03
18,Secret Factory,4.41,5.32,21.80,1.16,72.14,Low,0.12
19,The Other Factory,4.59,1.41,17.59,0.97,72.14,Low,0.08


In [3]:
recs = recommend(scored, top_n=15)
recs[['Rank','Product','Current Factory','Candidate Factory','Orders','Lead Reduction %','Distance Reduction %','Profit Impact %','Profit Stability %','Confidence','Risk Level','Score']].round(2)

,Rank,Product,Current Factory,Candidate Factory,Orders,Lead Reduction %,Distance Reduction %,Profit Impact %,Profit Stability %,Confidence,Risk Level,Score
0,1,Fun Dip,Sugar Shack,Secret Factory,3,8.15,50.19,6.52,100.00,45.14,High,0.35
1,2,Laffy Taffy,Sugar Shack,Wicked Choccy's,10,3.69,28.53,4.61,80.00,45.48,High,0.23
2,3,SweeTARTS,Sugar Shack,Secret Factory,10,5.08,25.02,3.32,90.00,54.48,Medium,0.19
3,4,Nerds,Sugar Shack,Secret Factory,4,5.62,28.84,3.32,100.00,46.48,High,0.19
4,5,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Secret Factory,1810,6.38,29.93,1.62,69.50,72.14,Medium,0.15
5,6,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Secret Factory,1818,6.51,30.61,1.57,69.97,72.14,Medium,0.15
6,7,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Secret Factory,2064,6.35,28.94,1.49,69.19,72.14,Medium,0.15
7,8,Fizzy Lifting Drinks,Sugar Shack,Secret Factory,6,5.75,32.21,1.43,100.00,49.14,High,0.13
8,9,Wonka Bar - Milk Chocolate,Wicked Choccy's,Secret Factory,2137,5.32,21.80,1.16,87.69,72.14,Low,0.12
9,10,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Secret Factory,2015,5.47,22.08,1.10,89.38,72.14,Low,0.11


In [4]:
kpis = network_kpis(scored, recs)
pd.Series(kpis).round(2)

Lead Time Reduction % (network)                   5.85
Lead Time Reduction % (reassigned products)       5.96
Profit Impact Stability %                        77.54
Scenario Confidence Score                        72.14
Recommendation Coverage % (products)             80.00
Recommendation Coverage % (orders)               97.87
Products Recommended                             12.00
Products Evaluated                               15.00
Network Profit Impact ($)                      1232.91
dtype: float64

### Sensitivity to the speed-vs-profit slider

In [5]:
rows = []
for w in [0, .25, .5, .75, 1.0]:
    s = score_scenarios(sim.simulate(), speed_weight=w); r = recommend(s, top_n=15); k = network_kpis(s, r)
    rows.append({'speed weight': w, 'products recommended': k['Products Recommended'], 'network lead reduction %': round(k['Lead Time Reduction % (network)'], 2), 'network profit impact $': round(k['Network Profit Impact ($)'])})
pd.DataFrame(rows)

,speed weight,products recommended,network lead reduction %,network profit impact $
0,0.00,12,5.85,1233
1,0.25,12,5.85,1233
2,0.50,12,5.85,1233
3,0.75,12,5.85,1233
4,1.00,12,5.85,1232


In [6]:
scored.to_csv(C.PROCESSED / 'scenario_simulation.csv', index=False)
recs.to_csv(C.PROCESSED / 'recommendations.csv', index=False)
alternatives_per_product(scored, 3).to_csv(C.PROCESSED / 'alternatives_per_product.csv', index=False)